# MSPM0L2228 — VCORE voltage glitch (ChipWhisperer-Lite)

Drives `firmware/` (Astra's unrolled `a += 1` x200 detector). Clean run => `A=200`;
almost any landed fault => `A != 200`.

**Before running**
- Hardware mod: C3 removed, replaced with ~480 nF + ~36 Ohm in series at the VCORE pad
  (Astra's best combo). Crowbar LP on the VCORE side. See `firmware/README.md`.
- **Crowbar safety: LP FET only, HP never** (asserted in the setup cell).
- Connection (eCTF): **XDS110** debugger, 8 jumpers to the board header; flash with **OpenOCD**.
  The host serial is the **XDS110 backchannel UART on PA10/PA11 (UART0)** -- no separate dongle.
- **nRST ownership:** after flashing, remove the XDS110 NRST (and SWDIO/SWCLK) jumpers and let
  the **CW drive NRST**; keep XDS110 3V3/GND/RXD/TXD for the backchannel serial. Share GND
  across board + CW + XDS110. (Board-to-board challenge comms is UART1/PA8/PA9 -- not used here.)
- Set `TARGET_PORT` below to the **XDS110 backchannel COM port** (re-check every session).

**Method (CLAUDE.md):** baseline gate -> positive control -> measure the window ->
sweep `ext_offset` across the WHOLE window (coarse->fine) and `repeat` from LOW. Timing
units are CW clkgen counts (~16.7 ns @ 60 MHz); the target self-clocks on its 32 MHz SYSOSC.
Astra's useful widths (~1-3 target cycles, cap ~20) are converted to `repeat` in cell 1.
Per-reset latch guard halts the run on a suspected SCR latch (only a USB power-cycle clears it).

In [ ]:
import time, chipwhisperer as cw

scope = cw.scope()
scope.default_setup()

# clkgen is the glitch time base (target self-clocks on its own 32 MHz SYSOSC)
scope.clock.clkgen_freq = 60e6
CW_CLK = scope.clock.clkgen_freq            # ~60 MHz -> ~16.7 ns per repeat/offset count

scope.glitch.clk_src     = 'clkgen'
scope.glitch.output      = 'enable_only'    # sustained crowbar; re-run later with 'glitch_only'
scope.glitch.trigger_src = 'ext_single'
scope.glitch.repeat      = 1
scope.glitch.ext_offset  = 0

# trigger: firmware raises PB0 -> wired to tio4
scope.trigger.triggers = 'tio4'
scope.io.tio4          = 'high_z'

# ---- crowbar SAFETY: LP FET only, HP never ----
scope.io.glitch_lp = True
scope.io.glitch_hp = False
assert scope.io.glitch_lp and not scope.io.glitch_hp, 'LP must be ON, HP OFF'

def ns_to_repeat(ns):
    return max(1, round(ns * 1e-9 * CW_CLK))

# Astra scaled to 32 MHz: ~1-3 cycles = ~31-94 ns; keep under ~20 cycles (~625 ns)
print('repeat ~31ns:', ns_to_repeat(31), ' ~94ns:', ns_to_repeat(94), ' cap ~625ns:', ns_to_repeat(625))
print(scope)

In [ ]:
import serial

TARGET_PORT = 'COM7'      # <-- XDS110 backchannel serial COM port (GetPortNames())
BAUD = 115200
ser = serial.Serial(TARGET_PORT, BAUD, timeout=0.2)

def reset_board(hold=0.05, settle=0.05):
    scope.io.nrst = 'low'
    time.sleep(hold)
    scope.io.nrst = 'high_z'
    time.sleep(settle)

def read_line(timeout=0.3):
    ser.timeout = timeout
    try:
        return ser.readline().decode(errors='ignore').strip()
    except Exception:
        return ''

def wait_boot(timeout=0.6):
    # reset, then confirm a clean reboot by seeing the BOOT banner (latch guard primitive)
    ser.reset_input_buffer()
    reset_board()
    t0 = time.time()
    while time.time() - t0 < timeout:
        if 'BOOT' in read_line(0.1):
            return True
    return False

def parse_a(line):
    if line.startswith('A='):
        try:
            return int(line[2:])
        except ValueError:
            return None
    return None

print('UART open on', TARGET_PORT)

In [ ]:
# ---- BASELINE GATE (CLAUDE.md): A=200 must be rock-solid before any glitching ----
N = 30
ok = 0
for i in range(N):
    if not wait_boot():
        print('reboot FAILED at', i, '-- fix wiring/power/port first'); break
    ser.reset_input_buffer()
    ser.write(b'g')
    a = parse_a(read_line(0.3))
    if a == 200:
        ok += 1
    else:
        print('baseline non-clean:', a)
print('baseline clean %d/%d' % (ok, N))
assert ok == N, 'Baseline unstable -- STOP, do not sweep.'

In [ ]:
# ---- MEASURE THE WINDOW + POSITIVE CONTROL ----
# window = trigger-high width in CW counts == the span to sweep ext_offset over
wait_boot()
ser.reset_input_buffer()
scope.arm()
ser.write(b'g')
timed_out = scope.capture()            # False means the trigger was seen
a = parse_a(read_line(0.3))
width = scope.adc.trig_count
print('trigger seen:', not timed_out, ' A=', a, ' window(trig_count)=', width)
OFFSET_MAX = int(width)

# positive control (lite): 't' forces a miscount -> MUST read as non-200 (proves detection)
wait_boot(); ser.reset_input_buffer(); ser.write(b't')
a = parse_a(read_line(0.3))
print('self-test A=', a, '->', 'DETECT OK' if (a is not None and a != 200) else 'DETECT BROKEN')

In [ ]:
# ---- SWEEP: ext_offset across the whole window (coarse) x repeat from LOW ----
import os, csv, json, datetime

mode = scope.glitch.output                  # re-run this cell with 'glitch_only' too
ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
run_dir = os.path.join('runs', ts + '_mspm0_vcore')
os.makedirs(run_dir, exist_ok=True)
json.dump({'cw_clk': CW_CLK, 'mode': mode, 'offset_max': OFFSET_MAX,
           'target_port': TARGET_PORT, 'expected_a': 200},
          open(os.path.join(run_dir, 'meta.json'), 'w'), indent=2)

OFFSETS = range(0, OFFSET_MAX + 1, 5)       # coarse; re-sweep hot spans at step 1
REPEATS = range(1, 31)                       # ~1..30 CW counts (~17..500 ns)

def classify():
    ser.reset_input_buffer()
    if not wait_boot():
        return ('LATCH?', None, None, '')
    ser.reset_input_buffer()
    scope.arm()
    ser.write(b'g')
    timed_out = scope.capture()
    raw = read_line(0.3)
    a = parse_a(raw)
    fired = (not timed_out)
    if a is None:
        return ('RESET', None, fired, raw)
    if a == 200:
        return ('CLEAN', a, fired, raw)
    return ('GLITCH', a, fired, raw)

counts = {}
rows = []
halt = False
for off in OFFSETS:
    if halt: break
    scope.glitch.ext_offset = int(off)
    for rep in REPEATS:
        scope.glitch.repeat = int(rep)
        result, a, fired, raw = classify()
        # per-reset LATCH GUARD: a RESET must reboot clean; if not -> SCR latch
        if result in ('RESET', 'LATCH?'):
            if not wait_boot():
                print('*** LATCH at off=%d rep=%d -- nRST will NOT clear it.' % (off, rep))
                print('*** USB POWER-CYCLE the board, then re-run. Halting.')
                result = 'LATCH?'; halt = True
            elif result == 'LATCH?':
                result = 'RESET'
        counts[result] = counts.get(result, 0) + 1
        rows.append([off, rep, mode, result, a, fired, raw])
        if result == 'GLITCH':
            print('GLITCH off=%d rep=%d A=%s' % (off, rep, a))
        if halt: break

with open(os.path.join(run_dir, 'trials.csv'), 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['ext_offset','repeat','mode','result','a_value','trig_fired','raw'])
    w.writerows(rows)
print('done', counts, '->', run_dir)

In [ ]:
# ---- hot spots: where did faults land? (map these back to the instruction stream) ----
g = [r for r in rows if r[3] == 'GLITCH']
print('total glitches:', len(g))
for r in sorted(set((row[0], row[1], row[4]) for row in g)):
    print('  off=%4d rep=%3d -> A=%s' % r)
print('Next: re-sweep hot offset spans at step=1; try output=\'glitch_only\'; '
      'rescan onset every ~30 min (thermal drift).')